# Identificação de Operadores Ineficientes
**Empresa:** CallMeMaybe (Serviço de Telefonia Virtual)

**Autor:** Fabrício Camacho

**Data:** 11/04/2026

## 1. Introdução e Contexto do Negócio

O serviço de telefonia virtual **CallMeMaybe** tem como foco atender clientes corporativos (organizações) que precisam gerenciar grandes volumes de chamadas diárias, sejam elas realizadas ou recebidas por diversos operadores. Além do contato externo, o sistema também suporta chamadas internas, feitas entre os próprios operadores usando a rede da empresa.

Atualmente, a empresa está desenvolvendo uma nova funcionalidade estratégica: um sistema focado em permitir que os supervisores identifiquem com precisão quais operadores estão apresentando baixa eficiência. Resolver este problema de negócio gerará valor ao otimizar a força de trabalho, melhorar a satisfação dos clientes finais (reduzindo tempo de espera e perdas) e direcionar treinamentos de forma mais assertiva.

### 1.1 Critérios de Ineficiência

Para garantir que a nossa análise seja objetiva e alinhada às expectativas dos *stakeholders*, um operador será classificado como ineficiente se apresentar um ou mais dos seguintes comportamentos:

- **Alto volume de perdas:** Possui muitas chamadas recebidas perdidas, independentemente de serem internas ou externas.
- **Lentidão no atendimento:** Apresenta um tempo de espera significativamente prolongado nas chamadas recebidas.
- **Baixa produtividade ativa:** No caso de operadores designados primariamente para chamadas de saída, o profissional realiza um número muito baixo de chamadas ativas.

### 1.2 Objetivo do Projeto

O escopo principal deste projeto consiste em realizar uma análise exploratória robusta dos dados de telefonia para identificar de forma sistemática esses operadores ineficientes. Paralelamente, testaremos hipóteses estatísticas para validar se fatores como o plano tarifário ou o tipo de chamada influenciam nessa performance, fornecendo uma base sólida para a tomada de decisão.

## 2. Descrição dos Dados e Dicionário de Variáveis

Para realizar esta análise, trabalharemos com dois conjuntos de dados principais que contêm informações sobre o uso do serviço de telefonia virtual e os planos dos clientes. 

Antes de qualquer manipulação, é fundamental compreender a estrutura de cada tabela e o que cada coluna representa.

### 2.1 Dataset de Chamadas (`telecom_dataset_us.csv`)
Este conjunto de dados contém os registros detalhados das ligações.

- **`user_id`**: ID da conta do cliente (organização);
- **`date`**: Data em que as estatísticas foram coletadas;
- **`direction`**: Direção da chamada (`out` para chamadas de saída, `in` para chamadas recebidas);
- **`internal`**: Indica se a chamada foi interna (entre operadores de um mesmo cliente);
- **`operator_id`**: Identificador único do operador;
- **`is_missed_call`**: Variável booleana que indica se foi uma chamada perdida;
- **`calls_count`**: Número de chamadas realizadas/recebidas no período;
- **`call_duration`**: Duração efetiva da chamada (não inclui o tempo de espera);
- **`total_call_duration`**: Duração total da chamada (incluindo o tempo de espera).

### 2.2 Dataset de Clientes (`telecom_clients_us.csv`)
Este conjunto de dados traz o perfil contratual de cada organização.

- **`user_id`**: ID do cliente (usado como chave para cruzar com a tabela de chamadas);
- **`tariff_plan`**: Plano tarifário atual do cliente[cite: 116];
- **`date_start`**: Data de registro (início do contrato) do cliente[cite: 117].

In [2]:
# Importando as bibliotecas necessárias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configurando o estilo dos gráficos
sns.set_theme(style="whitegrid")

# Carregando os conjuntos de dados
try:
    df_calls = pd.read_csv('datasets/telecom_dataset_new.csv')
    df_clients = pd.read_csv('datasets/telecom_clients.csv')
    print("Bases de dados carregadas com sucesso!")
except FileNotFoundError:
    print("Erro: Arquivos não encontrados. Verifique o diretório.")

# Visualizando as primeiras linhas e informações gerais do dataset de chamadas
display(df_calls.head())
print("\n--- Informações: Tabela de Chamadas ---")
df_calls.info()

print("\n" + "="*50 + "\n")

# Visualizando as primeiras linhas e informações gerais do dataset de clientes
display(df_clients.head())
print("\n--- Informações: Tabela de Clientes ---")
df_clients.info()


Bases de dados carregadas com sucesso!


,user_id,date,direction,internal,operator_id,is_missed_call,calls_count,call_duration,total_call_duration
0,166377,2019-08-04 00:00:00+03:00,in,False,NaN,True,2,0,4
1,166377,2019-08-05 00:00:00+03:00,out,True,880022.0,True,3,0,5
2,166377,2019-08-05 00:00:00+03:00,out,True,880020.0,True,1,0,1
3,166377,2019-08-05 00:00:00+03:00,out,True,880020.0,False,1,10,18
4,166377,2019-08-05 00:00:00+03:00,out,False,880022.0,True,3,0,25



--- Informações: Tabela de Chamadas ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53902 entries, 0 to 53901
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   user_id              53902 non-null  int64  
 1   date                 53902 non-null  object 
 2   direction            53902 non-null  object 
 3   internal             53785 non-null  object 
 4   operator_id          45730 non-null  float64
 5   is_missed_call       53902 non-null  bool   
 6   calls_count          53902 non-null  int64  
 7   call_duration        53902 non-null  int64  
 8   total_call_duration  53902 non-null  int64  
dtypes: bool(1), float64(1), int64(4), object(3)
memory usage: 3.3+ MB




,user_id,tariff_plan,date_start
0,166713,A,2019-08-15
1,166901,A,2019-08-23
2,168527,A,2019-10-29
3,167097,A,2019-09-01
4,168193,A,2019-10-16



--- Informações: Tabela de Clientes ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 732 entries, 0 to 731
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   user_id      732 non-null    int64 
 1   tariff_plan  732 non-null    object
 2   date_start   732 non-null    object
dtypes: int64(1), object(2)
memory usage: 17.3+ KB


## 3. Pré-processamento e Limpeza de Dados

Antes de extrairmos qualquer métrica, precisamos garantir a qualidade e a integridade dos dados. Nesta etapa, realizaremos as seguintes transformações:

1. **Conversão de Tipos:** As colunas que representam datas (`date` e `date_start`) serão convertidas para o formato `datetime` do Pandas, facilitando possíveis agregações temporais.
2. **Tratamento de Valores Nulos:** O foco principal do nosso problema de negócio é o desempenho do operador. Se uma chamada não possui um `operator_id` registrado, não podemos atribuí-la a nenhum funcionário. Portanto, avaliaremos a proporção desses dados ausentes e os removeremos para evitar distorções nas métricas individuais.
3. **Engenharia de Recursos (Feature Engineering):** Um dos critérios de ineficiência definidos pela CallMeMaybe é o tempo de espera prolongado nas chamadas recebidas. Os dados originais nos fornecem a duração total (`total_call_duration`) e a duração efetiva (`call_duration`). Criaremos uma nova variável chamada `waiting_time`, subtraindo a duração efetiva da duração total.

In [3]:
# 3.1 Conversão de Tipos (Datas)
df_calls['date'] = pd.to_datetime(df_calls['date'])
df_clients['date_start'] = pd.to_datetime(df_clients['date_start'])

# 3.2 Tratamento de Valores Ausentes
print("--- Contagem de Valores Nulos Antes da Limpeza ---")
display(df_calls.isnull().sum())

# Calculando a porcentagem de nulos na coluna operator_id
null_operator_pct = (df_calls['operator_id'].isnull().sum() / len(df_calls)) * 100
print(f"\nPorcentagem de chamadas sem operator_id: {null_operator_pct:.2f}%")

# Removendo linhas onde o 'operator_id' é nulo (pois inviabiliza a análise de ineficiência do operador)
# Usamos .copy() para evitar o aviso "SettingWithCopyWarning" do Pandas no futuro
df_calls_clean = df_calls.dropna(subset=['operator_id']).copy()

# Opcional: Converter operator_id para inteiro (caso ele tenha ficado como float devido aos nulos)
# Isso deixa o ID mais limpo visualmente (ex: 12345 em vez de 12345.0)
df_calls_clean['operator_id'] = df_calls_clean['operator_id'].astype(int)

# 3.3 Engenharia de Recursos (Criando a métrica de tempo de espera)
# Tempo de espera = Duração total da chamada - Duração efetiva da chamada
df_calls_clean['waiting_time'] = df_calls_clean['total_call_duration'] - df_calls_clean['call_duration']

# Verificando se as transformações foram aplicadas com sucesso
print("\n--- Amostra dos Dados Tratados ---")
display(df_calls_clean[['operator_id', 'total_call_duration', 'call_duration', 'waiting_time']].head())


--- Contagem de Valores Nulos Antes da Limpeza ---


user_id                   0
date                      0
direction                 0
internal                117
operator_id            8172
is_missed_call            0
calls_count               0
call_duration             0
total_call_duration       0
dtype: int64


Porcentagem de chamadas sem operator_id: 15.16%

--- Amostra dos Dados Tratados ---


,operator_id,total_call_duration,call_duration,waiting_time
1,880022,5,0,5
2,880020,1,0,1
3,880020,18,10,8
4,880022,25,0,25
5,880020,29,3,26
